# 04 - Restrictions

When Telegram restricts content, it does not hide the fact. It attaches a
`restriction_reason` to the message or channel: a list of objects, each naming a
**platform** the restriction applies on and a **reason** code. The client uses it
to decide whether to show the content to you.

This is a moderation record the platform itself publishes, per item, with a
machine-readable reason, and no prior general-purpose Telegram corpus collected
it. That is the contribution of this chapter: not a new inference method, but a
signal that was sitting in the protocol and going unrecorded.

**Figures produced here**

| Output | Thesis figure |
|---|---|
| `restriction_reasons_by_reason` | flag instances by reason |
| `restriction_reasons_by_platform` | flag instances by platform |
| `restriction_cooccurrence` | which reasons appear together on a channel |
| `restricted_lang_topic_raw` | language x reason, raw counts (representation) |
| `restricted_lang_topic_rate` | language x reason, per 1,000 messages (proneness) |

Every section here is *offline*: the aggregates are small enough to ship, so the
whole notebook renders without a database. The SQL that produced them is carried
anyway, since it is the record of what was counted.

**One reading rule for the whole notebook.** A *flag instance* is one
(channel, reason, platform) entry, weighted by how many messages carry it. It is
not a count of restricted channels, and the same channel appears under several
reasons. Counts and channel counts are reported side by side throughout for that
reason.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from _shared import DATA, FIGURES, csv, q, setup_plots

setup_plots()

## 1. What gets restricted, and where

The base frame unrolls the `restriction_reason` JSONB array into one row per
(channel, reason, platform), with the message count behind it.

Two things worth noticing in the reason vocabulary. The codes are
platform-specific by construction: `androidterms` and `appleterms` are the same
kind of finding routed through two different app-store policies, and the split
between them is a distribution artefact rather than a difference in what the
content is. And the platform axis is dominated by iOS and Android because those
are the stores whose review process forces Telegram's hand; `all` is the
Telegram-level restriction that applies everywhere.

Log scale on both bar charts: the top reasons outweigh the tail by four orders of
magnitude.

In [ ]:
SQL_RESTRICTION_UNROLL = """
    SELECT m.channel_id,
           elem->>'reason'   AS reason,
           elem->>'platform' AS platform,
           count(*)          AS flags
    FROM messages m, LATERAL jsonb_array_elements(m.restriction_reason) AS elem
    WHERE m.restriction_reason IS NOT NULL
    GROUP BY 1, 2, 3
"""
# by_reason / by_platform below are groupbys of that frame:
#   .agg(flag_instances=("flags", "sum"), channels=("channel_id", "nunique"))

by_reason = csv("restriction_reasons.csv")
print(f"reasons present: {len(by_reason)}   "
      f"total flag instances: {by_reason['flag_instances'].sum():,}")

fig, ax = plt.subplots(figsize=(5.48, max(1.74, 0.274 * len(by_reason))))
ax.barh(by_reason["reason"].astype(str), by_reason["flag_instances"], color="#7aa86f")
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_xlabel("flag instances (log)")
ax.set_title("Restriction flag instances by reason")
for i, (v, c) in enumerate(zip(by_reason["flag_instances"], by_reason["channels"])):
    ax.text(v, i, f"  {v:,} \u00b7 {c:,} ch", va="center", fontsize=7)
fig.tight_layout()
fig.savefig(FIGURES / "restriction_reasons_by_reason.png", bbox_inches="tight")
fig.savefig(FIGURES / "restriction_reasons_by_reason.pdf", bbox_inches="tight")
plt.show()
by_reason

In [ ]:
# Platform codes are given readable names; the two opaque low-volume codes
# (VIRT1, ms) are folded into "Other" rather than shown as raw identifiers.
PLAT_LABEL = {"ios": "iOS", "android": "Android", "all": "All platforms (Telegram-level)"}
PLAT_OTHER = {"VIRT1", "ms"}

by_platform = csv("signalB_by_platform.csv")
by_platform["platform"] = by_platform["platform"].astype(str)
by_platform["pl_label"] = np.where(
    by_platform["platform"].isin(PLAT_OTHER), "Other",
    by_platform["platform"].map(lambda p: PLAT_LABEL.get(p, p)))
by_platform_disp = (by_platform.groupby("pl_label", as_index=False)
                    .agg(flag_instances=("flag_instances", "sum"),
                         channels=("channels", "sum"))
                    .sort_values("flag_instances", ascending=False))

fig, ax = plt.subplots(figsize=(5.48, max(1.49, 0.30 * len(by_platform_disp))))
ax.barh(by_platform_disp["pl_label"].astype(str), by_platform_disp["flag_instances"],
        color="#5b8ef0")
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_xlabel("flag instances (log)")
ax.set_title("Restriction flag instances by platform")
for i, (v, c) in enumerate(zip(by_platform_disp["flag_instances"],
                               by_platform_disp["channels"])):
    ax.text(v, i, f"  {v:,} \u00b7 {c:,} ch", va="center", fontsize=7)
fig.tight_layout()
fig.savefig(FIGURES / "restriction_reasons_by_platform.png", bbox_inches="tight")
fig.savefig(FIGURES / "restriction_reasons_by_platform.pdf", bbox_inches="tight")
plt.show()
by_platform_disp

## 2. Reasons travel together

A restricted channel rarely carries one reason. The matrix below counts channels
carrying each pair of reasons, and colours each cell by the *conditional*
probability P(column reason | row reason), which is the asymmetric reading: a
channel flagged for the rare reason is very likely also flagged for the common
one, but not the reverse.

The diagonal is zeroed so it does not dominate the colour scale.

Reading the asymmetry is the point. The Apple and Android terms codes co-occur
heavily, which is the same underlying finding appearing under two store policies.
The narrower reasons sit almost entirely inside the broader ones.

In [ ]:
# From the unrolled frame: presence matrix (channel x reason), then its Gram
# matrix gives co-occurrence counts.
#
#   pres = rp.assign(v=1).pivot_table(index="channel_id", columns="reason",
#                                     values="v", aggfunc="max", fill_value=0)
#   cooc = pres.T.dot(pres); np.fill_diagonal(cooc.values, 0)
#   cond = cooc.div(pres.sum(axis=0), axis=0)      # P(col | row)

cooc = csv("restriction_reason_cooccurrence.csv", index_col=0)
# Channels carrying each reason: the co-occurrence row maxima understate this, so
# the conditional denominator comes from the reason table.
per_reason = csv("restriction_reasons.csv").set_index("reason")["channels"]
cond = cooc.div(per_reason.reindex(cooc.index).replace(0, np.nan), axis=0)

fig, ax = plt.subplots(figsize=(5.48, 4.26))
sns.heatmap(cond, annot=cooc.values, fmt=",d", cmap="Reds", ax=ax,
            vmin=0, vmax=1, annot_kws={"size": 6},
            cbar_kws={"label": "P(col reason | row reason)"})
ax.set_title("Reason co-occurrence: colour P(col|row), labels = channels with both")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()
fig.savefig(FIGURES / "restriction_cooccurrence.png", bbox_inches="tight")
fig.savefig(FIGURES / "restriction_cooccurrence.pdf", bbox_inches="tight")
plt.show()

## 3. Representation is not proneness

Crossing restriction reason with channel language produces two different figures
from the same table, and conflating them is the easy mistake.

The **raw** heatmap is representation: which language and reason combinations
carry the most flags. English and German dominate it, and a large part of that is
simply that those languages have a lot of messages in this corpus.

The **rate** heatmap normalises by the message volume of each language, giving
flag instances per 1,000 messages. That is proneness: given a message in this
language, how likely is it to sit under a restriction flag. The ordering changes
substantially, which is exactly why both are printed.

The denominator is every message in every channel whose modal language is that
language, so the two panels share a channel-language assignment (notebook 02,
section 7).

In [ ]:
SQL_CHANNEL_MODAL_LANG = """
    SELECT channel_id, lang FROM (
        SELECT channel_id, lang,
               row_number() OVER (PARTITION BY channel_id ORDER BY count(*) DESC) AS rk
        FROM message_languages GROUP BY channel_id, lang
    ) t WHERE rk = 1
"""
# The rate panel's denominator is the per-language message total, taken by joining
# that modal-language table to the per-channel message counts of notebook 01:
#
#   lang_msgs = chan_lang.merge(counts).groupby("lang")["messages"].sum()
#   rate      = mat.div(lang_msgs / 1000.0, axis=0)

mat = csv("restriction_by_language.csv", index_col=0)
rate = csv("restriction_rate_by_language.csv", index_col=0)

# Cells 1.3x taller than the textwidth default, so the annotations stay readable.
_lh = max(2.59, 0.356 * len(mat))

fig, ax = plt.subplots(figsize=(5.48, _lh))
sns.heatmap(np.log10(mat + 1), annot=mat.values, fmt=",d", cmap="viridis", ax=ax,
            annot_kws={"size": 6}, cbar_kws={"label": "log10(flag instances)"})
ax.set_title("Restriction by language x reason: raw flag instances (representation)")
ax.set_xlabel("reason")
ax.set_ylabel("modal language")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()
fig.savefig(FIGURES / "restricted_lang_topic_raw.png", bbox_inches="tight")
fig.savefig(FIGURES / "restricted_lang_topic_raw.pdf", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5.48, _lh))
sns.heatmap(np.log10(rate + 1e-3), annot=rate.values, fmt=".1f", cmap="rocket_r", ax=ax,
            annot_kws={"size": 6}, cbar_kws={"label": "log10(flags per 1,000 msgs)"})
ax.set_title("Restriction rate: flag instances per 1,000 messages of that language (proneness)")
ax.set_xlabel("reason")
ax.set_ylabel("modal language")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()
fig.savefig(FIGURES / "restricted_lang_topic_rate.png", bbox_inches="tight")
fig.savefig(FIGURES / "restricted_lang_topic_rate.pdf", bbox_inches="tight")
plt.show()
rate.round(2)

## What is not in this notebook

Two restriction outputs from the wider analysis are deliberately absent.

The **content samples** of restricted messages are user message text. They are not
needed for any figure in the thesis and are not redistributable, so they are not
shipped and no section reads them.

The **per-channel concentration** view (a Lorenz curve over how flags pile up on a
few channels) is not a thesis figure. Its input is the per-channel flagged-message
table, which is a channel-identifier list, and it is left out under the same rule
that keeps this repository to what backs a published result.

## Reading

Restriction in this corpus is overwhelmingly app-store driven. The two largest
reason codes are the Android and Apple terms-of-service findings, and the two
largest platform buckets are the two mobile stores; Telegram-level restriction,
which applies to everyone, is an order of magnitude smaller. What the flag
records is mostly Telegram complying with a distributor, not Telegram
moderating.

The reason axis is not a taxonomy of harm. `androidterms` and `appleterms` carry
no information about what the content is, only about which policy caught it, and
they co-occur heavily. The reasons that do name a category (`porn`, `copyright`,
`doxxing`, `violence`) are the interpretable ones, and they are much rarer.

Normalising by language volume reorders the picture enough that the raw heatmap
should not be read alone. A language can be heavily represented among flags
simply by being heavily represented in the corpus.

Finally, this is a signal available to anyone who asks the API for it. That it had
not been collected before is the finding; measuring it is the easy part.